In [1]:

!git clone https://github.com/sarthak-jain37/CSD361-Project.git



Cloning into 'CSD361-Project'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 14 (delta 2), reused 14 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 190.04 KiB | 9.50 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [2]:

from pathlib import Path

PROJECT_ROOT = Path("/content/CSD361-Project")

print("Repository found:", PROJECT_ROOT.exists())

for item in PROJECT_ROOT.iterdir():
    print(item.name)



Repository found: True
data
notebooks
.git
.gitignore


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("/content/CSD361-Project")

DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = DATA_DIR / "splits"

print("Feature Selection Environment Ready!")

Feature Selection Environment Ready!


In [4]:
manifest = pd.read_csv(DATA_DIR / "breakhis_manifest.csv")

image_folds = pd.read_csv(
    SPLITS_DIR / "image_level" / "image_level_folds.csv"
)

patient_folds = pd.read_csv(
    SPLITS_DIR / "patient_level" / "patient_level_folds.csv"
)

print("Manifest loaded:", len(manifest))
print("Image-level folds:", image_folds.shape)
print("Patient-level folds:", patient_folds.shape)

Manifest loaded: 7663
Image-level folds: (7663, 9)
Patient-level folds: (7663, 9)



# Feature Selection Objective

The objective of this stage is to identify the most informative features for distinguishing benign and malignant breast histopathology images.

Feature selection will be performed after feature extraction and before model training.

The selection process will:

- Remove irrelevant and redundant features.
- Identify informative features.
- Reduce feature dimensionality where appropriate.
- Avoid data leakage by fitting selection methods only on training folds.

The selected features will be passed to the model training stage.


In [5]:

# Import required libraries

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set project directory
PROJECT_ROOT = Path("/content/CSD361-Project")

# Define data directories
DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = DATA_DIR / "splits"

print("Libraries imported successfully!")
print("Project directory:", PROJECT_ROOT)
print("Project directory exists:", PROJECT_ROOT.exists())


Libraries imported successfully!
Project directory: /content/CSD361-Project
Project directory exists: True



## Project Data

This project uses the preprocessed BreakHis dataset prepared in the previous stage.

The existing preprocessing outputs include:

- Image manifest containing image metadata and binary labels.
- Image-level five-fold cross-validation splits.
- Patient-level five-fold cross-validation splits.

These existing files will be reused for the feature selection experiments.


In [6]:

# Load the existing cross-validation splits

image_folds = pd.read_csv(
    SPLITS_DIR / "image_level" / "image_level_folds.csv"
)

patient_folds = pd.read_csv(
    SPLITS_DIR / "patient_level" / "patient_level_folds.csv"
)

print("Image-level folds loaded successfully!")
print("Patient-level folds loaded successfully!")


Image-level folds loaded successfully!
Patient-level folds loaded successfully!



## Experimental Approach

Feature selection will follow the cross-validation strategy established during preprocessing.

Two experimental settings will be considered:

1. **Image-level cross-validation:** Five folds, allowing images from the same candidate patient group to appear in different folds.

2. **Patient-level cross-validation:** Five folds, keeping candidate patient groups separate between training and testing data.

Feature selection will be fitted only on the training portion of each fold to prevent data leakage.


In [7]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/BreakHis_Project")

for item in DRIVE_ROOT.iterdir():
    print(item.name)


Folds.csv
BreaKHis_v1


In [9]:

DATASET_ROOT = DRIVE_ROOT / "BreaKHis_v1"

print("Folder exists:", DATASET_ROOT.exists())

for path in DATASET_ROOT.iterdir():
    print(path.name)


Folder exists: True
BreaKHis_v1


In [10]:

image_files = list(DATASET_ROOT.rglob("*.png"))

print("Total PNG images found:", len(image_files))
print("\nSample image paths:")
for path in image_files[:5]:
    print(path)


Total PNG images found: 7909

Sample image paths:
/content/drive/MyDrive/BreakHis_Project/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign/SOB/tubular_adenoma/SOB_B_TA_14-19854C/100X/SOB_B_TA-14-19854C-100-005.png
/content/drive/MyDrive/BreakHis_Project/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign/SOB/tubular_adenoma/SOB_B_TA_14-19854C/100X/SOB_B_TA-14-19854C-100-009.png
/content/drive/MyDrive/BreakHis_Project/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign/SOB/tubular_adenoma/SOB_B_TA_14-19854C/100X/SOB_B_TA-14-19854C-100-011.png
/content/drive/MyDrive/BreakHis_Project/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign/SOB/tubular_adenoma/SOB_B_TA_14-19854C/100X/SOB_B_TA-14-19854C-100-008.png
/content/drive/MyDrive/BreakHis_Project/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign/SOB/tubular_adenoma/SOB_B_TA_14-19854C/100X/SOB_B_TA-14-19854C-100-002.png


In [11]:

from google.colab import drive
from pathlib import Path
import subprocess

# Reconnect Google Drive
drive.mount('/content/drive')

# Restore GitHub repository
PROJECT_ROOT = Path("/content/CSD361-Project")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/sarthak-jain37/CSD361-Project.git",
         str(PROJECT_ROOT)],
        check=True
    )

# Restore dataset path
DATASET_ROOT = (
    Path("/content/drive/MyDrive/BreakHis_Project")
    / "BreaKHis_v1" / "BreakHis_v1"
)

print("GitHub project exists:", PROJECT_ROOT.exists())
print("BreakHis dataset exists:", DATASET_ROOT.exists())



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GitHub project exists: True
BreakHis dataset exists: False


In [12]:

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/BreakHis_Project")

for p in DRIVE_ROOT.rglob("histology_slides"):
    print("FOUND IMAGE FOLDER:", p)


FOUND IMAGE FOLDER: /content/drive/MyDrive/BreakHis_Project/BreaKHis_v1/BreaKHis_v1/histology_slides


In [13]:

DATASET_ROOT = (
    Path("/content/drive/MyDrive/BreakHis_Project")
    / "BreaKHis_v1"
    / "BreaKHis_v1"
)

manifest_path = PROJECT_ROOT / "data" / "processed" / "breakhis_manifest.csv"
manifest_df = pd.read_csv(manifest_path)

relative_path = manifest_df.loc[0, "image_path"]
sample_path = DATASET_ROOT / relative_path

print("Dataset folder exists:", DATASET_ROOT.exists())
print("Manifest rows:", len(manifest_df))
print("Sample image exists:", sample_path.is_file())
print("Sample path:", sample_path)


Dataset folder exists: True
Manifest rows: 7663
Sample image exists: False
Sample path: /content/drive/MyDrive/BreakHis_Project/BreaKHis_v1/BreaKHis_v1/archive (2)/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign/SOB/adenosis/SOB_B_A_14-22549AB/100X/SOB_B_A-14-22549AB-100-001.png


In [14]:

DATASET_ROOT = (
    Path("/content/drive/MyDrive/BreakHis_Project")
    / "BreaKHis_v1"
    / "BreaKHis_v1"
    / "archive (2)"
    / "BreaKHis_v1"
    / "BreaKHis_v1"
)

relative_path = manifest_df.loc[0, "image_path"]
sample_path = DATASET_ROOT / relative_path

print("Dataset folder exists:", DATASET_ROOT.exists())
print("Manifest rows:", len(manifest_df))
print("Sample image exists:", sample_path.is_file())


Dataset folder exists: False
Manifest rows: 7663
Sample image exists: False


In [15]:

from pathlib import Path

drive_project = Path("/content/drive/MyDrive/BreakHis_Project")
relative_path = Path(str(manifest_df.loc[0, "image_path"]))

matches = list(drive_project.rglob(relative_path.name))

print("Manifest image:", relative_path.name)
print("Number of matches:", len(matches))

for match in matches:
    print("Exact location:", match)


Manifest image: SOB_B_A-14-22549AB-100-001.png
Number of matches: 1
Exact location: /content/drive/MyDrive/BreakHis_Project/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign/SOB/adenosis/SOB_B_A_14-22549AB/100X/SOB_B_A-14-22549AB-100-001.png


In [16]:

from pathlib import Path
from collections import defaultdict

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/BreakHis_Project"
)

# Find all original images
actual_images = list(DRIVE_PROJECT.rglob("*.png"))

# Index images by their path starting at histology_slides
image_index = defaultdict(list)

for img in actual_images:
    parts = img.parts
    if "histology_slides" in parts:
        i = parts.index("histology_slides")
        key = Path(*parts[i:])
        image_index[key].append(img)

# Match manifest paths using the same relative suffix
matched_paths = []
missing = []

for relative in manifest_df["image_path"]:
    parts = Path(relative).parts
    if "histology_slides" not in parts:
        missing.append(relative)
        matched_paths.append(None)
        continue

    i = parts.index("histology_slides")
    key = Path(*parts[i:])
    matches = image_index.get(key, [])

    if len(matches) == 1:
        matched_paths.append(matches[0])
    else:
        matched_paths.append(None)
        missing.append(relative)

print("Manifest images:", len(manifest_df))
print("Original images found:", len(actual_images))
print("Successfully matched:", sum(p is not None for p in matched_paths))
print("Missing or ambiguous:", len(missing))



Manifest images: 7663
Original images found: 7909
Successfully matched: 7663
Missing or ambiguous: 0



### Hardware Configuration

GPU acceleration is used to improve the computational efficiency of CNN-based feature extraction.

The pretrained ResNet50 model will process images in batches to manage memory usage efficiently. The extracted feature representations will be stored for subsequent feature selection experiments.


In [17]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
GPU available: True
GPU: Tesla T4



## 7. CNN Architecture: ResNet50

### 7.1 Overview

A Convolutional Neural Network (CNN) is used to extract meaningful visual representations from the BreakHis breast histopathology images.

In this project, ResNet50 is selected as the feature extraction architecture. ResNet50 is a 50-layer deep residual network that uses skip connections to facilitate the learning of deep visual representations.

### 7.2 Transfer Learning Approach

The implementation uses ResNet50 pretrained on the ImageNet dataset.

The pretrained network weights are kept frozen, meaning that the CNN is used as a feature extractor rather than being fine-tuned on the BreakHis dataset.

The original final classification layer is removed, and the output from the global average pooling layer is used as the extracted feature representation.

### 7.3 Feature Dimension

For each input image, ResNet50 produces a feature vector containing 2,048 numerical values.

These feature vectors will be used as input to the subsequent feature selection stage.


In [18]:

import torch
import torch.nn as nn
from torchvision import models

# Select GPU when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ImageNet-pretrained ResNet50
weights = models.ResNet50_Weights.DEFAULT
resnet50 = models.resnet50(weights=weights)

# Remove the final classification layer
feature_extractor = nn.Sequential(
    *list(resnet50.children())[:-1]
)

# Freeze all CNN parameters
for param in feature_extractor.parameters():
    param.requires_grad = False

# Set evaluation mode
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

print("Model: ResNet50")
print("Device:", device)
print("Feature dimension:", resnet50.fc.in_features)
print("CNN weights frozen:",
      all(not p.requires_grad for p in feature_extractor.parameters()))


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 165MB/s]


Model: ResNet50
Device: cuda
Feature dimension: 2048
CNN weights frozen: True



## 8. CNN-Based Feature Extraction

### 8.1 Image Preprocessing

The original BreakHis images are loaded using the image paths matched with the processed manifest.

Each image is converted to RGB format and resized to 224 × 224 pixels, which is the standard input size used by the pretrained ResNet50 architecture.

The images are normalized using the mean and standard deviation associated with the ImageNet pretrained weights.

### 8.2 Batch-Based Feature Extraction

To use GPU memory efficiently, images are processed in batches rather than loading the entire dataset into memory.

The frozen ResNet50 model processes each batch and generates a 2,048-dimensional feature vector for every image.

The extracted feature vectors are stored alongside their corresponding labels and metadata for subsequent feature selection.


In [19]:

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class BreakHisDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert("RGB")
        image = self.transform(image)
        return image, index

# ResNet50 ImageNet preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = BreakHisDataset(matched_paths, transform)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Images ready:", len(dataset))
print("Batch size:", loader.batch_size)


Images ready: 7663
Batch size: 32



## 9. Feature Extraction Using ResNet50

The pretrained ResNet50 model is used to transform each histopathology image into a numerical feature representation.

The final classification layer is excluded, and the output of the global average pooling layer is used as the deep feature vector.

Each image produces a vector of 2,048 features. Feature extraction is performed in batches using GPU acceleration.

To support reproducibility and avoid unnecessary computation, extracted batches are saved to Google Drive. This allows completed batches to be reused if the Colab runtime is interrupted.

The extracted features retain the same ordering as the processed manifest, allowing them to be associated with their original labels and metadata.


In [20]:

from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

# Save checkpoints permanently in Google Drive
FEATURE_DIR = (
    Path("/content/drive/MyDrive/BreakHis_Project")
    / "extracted_features"
    / "resnet50"
)

BATCH_DIR = FEATURE_DIR / "batches"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

# Extract features batch by batch
feature_extractor.eval()

for batch_num, (images, indices) in enumerate(tqdm(loader)):
    batch_file = BATCH_DIR / f"batch_{batch_num:04d}.npz"

    # Reuse completed batches after a runtime interruption
    if batch_file.exists():
        continue

    images = images.to(device)

    with torch.inference_mode():
        outputs = feature_extractor(images)
        outputs = outputs.flatten(1)

    batch_features = outputs.cpu().numpy()

    # Save features and their original manifest indices
    np.savez_compressed(
        batch_file,
        indices=indices.numpy(),
        features=batch_features
    )

print("Batch feature extraction completed.")


  0%|          | 0/240 [00:00<?, ?it/s]

Batch feature extraction completed.


In [21]:

# Reconstruct the complete feature matrix
features = np.zeros((len(manifest_df), 2048), dtype=np.float32)

batch_files = sorted(BATCH_DIR.glob("batch_*.npz"))

for batch_file in batch_files:
    data = np.load(batch_file)
    features[data["indices"]] = data["features"]

# Save the feature matrix
feature_file = FEATURE_DIR / "resnet50_features.npy"
np.save(feature_file, features)

# Save corresponding metadata
metadata = manifest_df.copy()
metadata["feature_index"] = np.arange(len(metadata))

metadata_file = FEATURE_DIR / "resnet50_metadata.csv"
metadata.to_csv(metadata_file, index=False)

print("Feature extraction summary")
print("--------------------------")
print("Images:", features.shape[0])
print("Features per image:", features.shape[1])
print("Feature matrix shape:", features.shape)
print("Saved features:", feature_file)
print("Saved metadata:", metadata_file)


Feature extraction summary
--------------------------
Images: 7663
Features per image: 2048
Feature matrix shape: (7663, 2048)
Saved features: /content/drive/MyDrive/BreakHis_Project/extracted_features/resnet50/resnet50_features.npy
Saved metadata: /content/drive/MyDrive/BreakHis_Project/extracted_features/resnet50/resnet50_metadata.csv



### 9.1 Feature Extraction Results

The pretrained ResNet50 architecture was used to extract deep visual features from the BreakHis histopathology images.

The final classification layer was removed, and the output of the global average pooling layer was used as the feature representation.

Each image was converted into a 2,048-dimensional feature vector.

The extracted feature matrix contains 7,663 image representations, resulting in a matrix of dimensions 7,663 × 2,048.

The feature matrix and corresponding metadata were saved separately in Google Drive for reuse during the feature selection stage.



## 10. Feature Selection

### 10.1 Objective

Although the pretrained ResNet50 architecture extracts 2,048 features from each image, not all extracted features may contribute equally to the classification task.

Feature selection aims to identify the most informative features while removing irrelevant or redundant features.

This process can reduce feature dimensionality, improve computational efficiency, and provide a more compact representation of the histopathology images.

### 10.2 Selected Feature Selection Techniques

The following approaches will be investigated:

1. **Mutual Information:** Measures the statistical dependency between individual features and the target class labels.
2. **Correlation-Based Filtering:** Identifies highly correlated features and removes redundant representations.
3. **Random Forest-Based Feature Importance:** Uses a tree-based model to estimate the relative importance of extracted features.

The selected features will subsequently be made available to the classification stage of the project.

### 10.3 Prevention of Data Leakage

Feature selection involves learning information from the dataset. Therefore, selection methods will be fitted using only the training portion of each cross-validation fold.

The fitted selection procedure will then be applied to the corresponding held-out fold without refitting.


In [22]:

# Load the saved feature matrix and metadata
features = np.load(feature_file)
metadata = pd.read_csv(metadata_file)

# Extract target labels
y = metadata["label"].to_numpy()

print("Feature matrix:", features.shape)
print("Target labels:", y.shape)
print("Benign samples:", np.sum(y == 0))
print("Malignant samples:", np.sum(y == 1))
print("Feature values finite:", np.isfinite(features).all())


Feature matrix: (7663, 2048)
Target labels: (7663,)
Benign samples: 2480
Malignant samples: 5183
Feature values finite: True



### 10.4 Feature Matrix Inspection

The extracted feature matrix was successfully loaded and inspected.

The dataset contains 7,663 samples, with each sample represented by 2,048 deep features extracted using ResNet50.

The target labels are consistent with the original preprocessing:
- Benign (0): 2,480 images
- Malignant (1): 5,183 images

All extracted feature values were verified to be finite, with no NaN or infinite values detected.

The feature matrix is therefore ready for the feature selection experiments.


In [23]:

# Load existing fold assignments
image_folds = pd.read_csv(
    PROJECT_ROOT / "data/processed/splits/image_level/image_level_folds.csv"
)

patient_folds = pd.read_csv(
    PROJECT_ROOT / "data/processed/splits/patient_level/patient_level_folds.csv"
)

print("Image-level folds:", image_folds.shape)
print("Patient-level folds:", patient_folds.shape)

print("\nImage-level columns:", image_folds.columns.tolist())
print("Patient-level columns:", patient_folds.columns.tolist())

print("\nImage-level fold counts:")
print(image_folds["fold_id"].value_counts().sort_index())

print("\nPatient-level fold counts:")
print(patient_folds["fold_id"].value_counts().sort_index())


Image-level folds: (7663, 9)
Patient-level folds: (7663, 9)

Image-level columns: ['image_path', 'filename', 'label', 'class', 'subtype', 'case_id', 'patient_group_id', 'magnification', 'fold_id']
Patient-level columns: ['image_path', 'filename', 'label', 'class', 'subtype', 'case_id', 'patient_group_id', 'magnification', 'fold_id']

Image-level fold counts:
fold_id
1    1533
2    1533
3    1533
4    1532
5    1532
Name: count, dtype: int64

Patient-level fold counts:
fold_id
1    1627
2    1560
3    1649
4    1414
5    1413
Name: count, dtype: int64



### 10.5 Cross-Validation Split Verification

The existing five-fold image-level and patient-level split assignments generated during the preprocessing stage are reused for feature selection.

The image-level splits provide a baseline evaluation protocol, while the patient-level splits ensure that filename-derived candidate patient groups do not overlap between training and held-out data.

Before feature selection, the split assignments are aligned with the extracted feature matrix using image paths. This ensures that each feature vector, target label, and fold assignment refers to the same image.

Feature selection is performed independently within each training fold to prevent data leakage.


In [24]:

# Verify that fold assignments match the feature matrix rows

def align_fold_assignments(split_df, split_name):
    assert split_df["image_path"].is_unique, (
        f"Duplicate image paths in {split_name}"
    )

    split_indexed = split_df.set_index("image_path")

    assert set(split_indexed.index) == set(manifest_df["image_path"]), (
        f"Image paths do not match in {split_name}"
    )

    aligned = split_indexed.loc[manifest_df["image_path"]]

    assert np.array_equal(
        aligned["label"].to_numpy(),
        metadata["label"].to_numpy()
    ), f"Label mismatch in {split_name}"

    return aligned["fold_id"].to_numpy()


image_fold_ids = align_fold_assignments(
    image_folds, "Image-level"
)

patient_fold_ids = align_fold_assignments(
    patient_folds, "Patient-level"
)

# Check that candidate patient groups do not cross
# patient-level training and held-out folds
group_ids = metadata["patient_group_id"].to_numpy()

for fold in range(1, 6):
    train_groups = set(group_ids[patient_fold_ids != fold])
    test_groups = set(group_ids[patient_fold_ids == fold])

    assert train_groups.isdisjoint(test_groups), (
        f"Candidate patient group leakage in fold {fold}"
    )

print("Image paths aligned: True")
print("Labels aligned: True")
print("Image-level folds:", np.unique(image_fold_ids))
print("Patient-level folds:", np.unique(patient_fold_ids))
print("Patient-level group separation: Verified")


Image paths aligned: True
Labels aligned: True
Image-level folds: [1 2 3 4 5]
Patient-level folds: [1 2 3 4 5]
Patient-level group separation: Verified


In [25]:

from sklearn.feature_selection import SelectKBest, mutual_info_classif

K_FEATURES = 200

# Test Mutual Information on training data from image-level fold 1
train_mask = image_fold_ids != 1
test_mask = image_fold_ids == 1

mi_selector = SelectKBest(
    score_func=mutual_info_classif,
    k=K_FEATURES
)

X_train_mi = mi_selector.fit_transform(
    features[train_mask],
    y[train_mask]
)

X_test_mi = mi_selector.transform(
    features[test_mask]
)

print("Original features:", features.shape[1])
print("Selected features:", X_train_mi.shape[1])
print("Training samples:", X_train_mi.shape[0])
print("Held-out samples:", X_test_mi.shape[0])


Original features: 2048
Selected features: 200
Training samples: 6130
Held-out samples: 1533


In [26]:

from sklearn.feature_selection import VarianceThreshold

def correlation_feature_indices(X_train, threshold=0.95, max_features=200):
    # Remove constant features
    variance_selector = VarianceThreshold(threshold=0.0)
    X_var = variance_selector.fit_transform(X_train)
    remaining = np.where(variance_selector.get_support())[0]

    # Rank remaining features by variance
    variances = np.var(X_var, axis=0)
    ranked = np.argsort(variances)[::-1]

    # Greedily remove highly correlated features
    selected = []

    for idx in ranked:
        if not selected:
            selected.append(idx)
        else:
            correlations = np.abs(
                np.corrcoef(X_var[:, selected].T, X_var[:, idx])[0:-1, -1]
            )
            if np.all(correlations < threshold):
                selected.append(idx)

        if len(selected) >= max_features:
            break

    return remaining[selected]


In [27]:

from sklearn.ensemble import RandomForestClassifier

def random_forest_feature_indices(
    X_train, y_train, max_features=200
):
    selector = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    selector.fit(X_train, y_train)

    importances = selector.feature_importances_
    selected = np.argsort(importances)[::-1][:max_features]

    return selected


In [28]:

import json
from pathlib import Path
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# Experiment configuration
K_FEATURES = 200
CORRELATION_THRESHOLD = 0.95
RANDOM_STATE = 42

SELECTION_DIR = FEATURE_DIR / "feature_selection"
SELECTION_DIR.mkdir(parents=True, exist_ok=True)


# Method 1: Mutual Information
def select_mutual_information(X_train, y_train):
    selector = SelectKBest(
        score_func=lambda X, y: mutual_info_classif(
            X, y, random_state=RANDOM_STATE
        ),
        k=K_FEATURES
    )
    selector.fit(X_train, y_train)
    return selector.get_support(indices=True)


# Method 2: Correlation-based filtering
def select_correlation(X_train):
    variances = np.var(X_train, axis=0)
    remaining = np.flatnonzero(variances > 0)

    if len(remaining) == 0:
        raise ValueError("No non-constant features found.")

    X_remaining = X_train[:, remaining]
    corr = np.corrcoef(X_remaining, rowvar=False)

    # Handle numerical issues
    corr = np.nan_to_num(corr, nan=0.0)
    np.fill_diagonal(corr, 0.0)

    ranked = np.argsort(variances[remaining])[::-1]
    selected_local = []

    for idx in ranked:
        if all(
            abs(corr[idx, chosen]) < CORRELATION_THRESHOLD
            for chosen in selected_local
        ):
            selected_local.append(idx)

        if len(selected_local) == K_FEATURES:
            break

    return remaining[np.array(selected_local)]


# Method 3: Random Forest feature importance
def select_random_forest(X_train, y_train):
    selector = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    )
    selector.fit(X_train, y_train)

    return np.argsort(selector.feature_importances_)[::-1][:K_FEATURES]


# Run selection independently for every fold and split protocol
split_protocols = {
    "image_level": image_fold_ids,
    "patient_level": patient_fold_ids
}

selection_methods = {
    "mutual_information": select_mutual_information,
    "correlation": select_correlation,
    "random_forest": select_random_forest
}

selection_summary = []

for protocol_name, fold_ids in split_protocols.items():

    for fold in range(1, 6):

        train_idx = np.flatnonzero(fold_ids != fold)
        test_idx = np.flatnonzero(fold_ids == fold)

        X_train = features[train_idx]
        X_test = features[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        for method_name, selection_function in selection_methods.items():

            print(
                f"Running {protocol_name} | Fold {fold} | {method_name}"
            )

            # Fit feature selection using training data only
            if method_name == "correlation":
                selected_indices = selection_function(X_train)
            else:
                selected_indices = selection_function(X_train, y_train)

            # Apply the same selected features to train and test data
            X_train_selected = X_train[:, selected_indices]
            X_test_selected = X_test[:, selected_indices]

            # Save fold-specific outputs
            output_file = (
                SELECTION_DIR
                / f"{protocol_name}_fold{fold}_{method_name}.npz"
            )

            np.savez_compressed(
                output_file,
                selected_indices=selected_indices,
                train_indices=train_idx,
                test_indices=test_idx,
                X_train=X_train_selected,
                X_test=X_test_selected,
                y_train=y_train,
                y_test=y_test
            )

            selection_summary.append({
                "protocol": protocol_name,
                "fold": fold,
                "method": method_name,
                "selected_features": len(selected_indices),
                "train_samples": len(train_idx),
                "test_samples": len(test_idx),
                "output_file": str(output_file)
            })

print("\nAll feature selection runs completed.")


Running image_level | Fold 1 | mutual_information
Running image_level | Fold 1 | correlation
Running image_level | Fold 1 | random_forest
Running image_level | Fold 2 | mutual_information
Running image_level | Fold 2 | correlation
Running image_level | Fold 2 | random_forest
Running image_level | Fold 3 | mutual_information
Running image_level | Fold 3 | correlation
Running image_level | Fold 3 | random_forest
Running image_level | Fold 4 | mutual_information
Running image_level | Fold 4 | correlation
Running image_level | Fold 4 | random_forest
Running image_level | Fold 5 | mutual_information
Running image_level | Fold 5 | correlation
Running image_level | Fold 5 | random_forest
Running patient_level | Fold 1 | mutual_information
Running patient_level | Fold 1 | correlation
Running patient_level | Fold 1 | random_forest
Running patient_level | Fold 2 | mutual_information
Running patient_level | Fold 2 | correlation
Running patient_level | Fold 2 | random_forest
Running patient_level 

In [29]:
selection_results = pd.DataFrame(selection_summary)

summary_file = SELECTION_DIR / "feature_selection_summary.csv"
selection_results.to_csv(summary_file, index=False)

print("Total feature selection runs:", len(selection_results))
print("\nRuns by protocol and method:")
print(selection_results.groupby(["protocol", "method"]).size())

print("\nSelected feature counts:")
print(
    selection_results.groupby("method")["selected_features"]
    .agg(["mean", "min", "max"])
)

print("\nSummary saved to:", summary_file)

Total feature selection runs: 30

Runs by protocol and method:
protocol       method            
image_level    correlation           5
               mutual_information    5
               random_forest         5
patient_level  correlation           5
               mutual_information    5
               random_forest         5
dtype: int64

Selected feature counts:
                     mean  min  max
method                             
correlation         200.0  200  200
mutual_information  200.0  200  200
random_forest       200.0  200  200

Summary saved to: /content/drive/MyDrive/BreakHis_Project/extracted_features/resnet50/feature_selection/feature_selection_summary.csv


In [30]:
# Verify all saved feature-selection outputs
saved_files = list(SELECTION_DIR.glob("*.npz"))

print("Expected output files: 30")
print("Actual output files:", len(saved_files))

assert len(saved_files) == 30, "Some output files are missing!"

# Check every saved result
for file in saved_files:
    data = np.load(file)

    assert data["X_train"].shape[1] == len(data["selected_indices"])
    assert data["X_test"].shape[1] == len(data["selected_indices"])
    assert len(np.intersect1d(
        data["train_indices"], data["test_indices"]
    )) == 0

print("All 30 output files verified successfully.")

Expected output files: 30
Actual output files: 30
All 30 output files verified successfully.


In [31]:

def load_selected_features(protocol, fold, method):
    """
    Load fold-specific features selected using training data only.

    protocol: 'image_level' or 'patient_level'
    fold: 1 to 5
    method: 'mutual_information', 'correlation', or 'random_forest'
    """

    file_path = (
        SELECTION_DIR
        / f"{protocol}_fold{fold}_{method}.npz"
    )

    data = np.load(file_path)

    return {
        "X_train": data["X_train"],
        "X_test": data["X_test"],
        "y_train": data["y_train"],
        "y_test": data["y_test"],
        "selected_indices": data["selected_indices"],
        "train_indices": data["train_indices"],
        "test_indices": data["test_indices"]
    }


# Example: load patient-level fold 1 using Mutual Information
example = load_selected_features(
    protocol="patient_level",
    fold=1,
    method="mutual_information"
)

print("Training features:", example["X_train"].shape)
print("Testing features:", example["X_test"].shape)
print("Selected feature count:", len(example["selected_indices"]))


Training features: (6036, 200)
Testing features: (1627, 200)
Selected feature count: 200
